# Advection — advection of a magnetic bump

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **both the 2D and the 3D** ``advection`` cases from a single
shared data template (`src/advection.data`). Per the TRUST validation-form
layout, other `MagneticStandalone` cases live in their own sibling folders.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son_dir(subdir, case_name, name):
    """Read probe `name` from an explicit case subdirectory under build/."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    assert os.path.exists(path), f"no probe file found: {path}"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)


## Run

Runs both dimensions on **2 processors** (`nbProcs=2` + `case.partition()`).
The shared template is substituted with the appropriate 2D or 3D mesh,
boundaries, velocity, and probes; the 3D mesh is converted with
`Polyedriser dom` before partitioning.


In [ ]:
from trustutils import run

NB_PROCS = 2
run.reset()

# --- 2D mesh-convergence scan (shared ladder) -------------------------------
NX0_2D = 65
meshes_2d = {
    "coarse":   {"nx": 65, "ny": 17},
    "standard": {"nx": 2 * NX0_2D - 1, "ny": 17},
    "fine":     {"nx": 4 * NX0_2D - 3, "ny": 17},
}
cases_2d = {}
for lvl, dic in meshes_2d.items():
    dd = {
        "dim": 2,
        "origine": "0.  0.",
        "nodes": f"{dic['nx']} {dic['ny']}",
        "lengths": "1.  1.",
        "borders": (
            "Bord gauche   X = 0.    0. <= Y <= 1.\n"
            "        Bord droite   X = 1.    0. <= Y <= 1.\n"
            "        Bord bas      Y = 0.    0. <= X <= 1.\n"
            "        Bord haut     Y = 1.    0. <= X <= 1."
        ),
        "polyedriser": "",
        "nvel": 2,
        "vel": "0.2 0.",
        "bc": (
            "gauche  Neumann_Paroi_adiabatique\n"
            "            droite  Neumann_Paroi_adiabatique\n"
            "            bas     Neumann_Paroi_adiabatique\n"
            "            haut    Neumann_Paroi_adiabatique"
        ),
        "probes": (
            "sonde_x015 grav magnetic_field Periode 1e-6 Points 1 0.15 0.5\n"
            "      sonde_x025 grav magnetic_field Periode 1e-6 Points 1 0.25 0.5\n"
            "      sonde_x045 grav magnetic_field Periode 1e-6 Points 1 0.45 0.5"
        ),
    }
    cases_2d[lvl] = run.addCaseFromTemplate("advection.data", f"advection_2d/{lvl}",
                                            dic=dd, nbProcs=NB_PROCS, excluNR=True)
    cases_2d[lvl].partition()

# --- 3D mesh-convergence scan (shared ladder) -------------------------------
NX0_3D = 65
meshes_3d = {
    "coarse":   {"nx": 65, "ny": 9, "nz": 9},
    "standard": {"nx": 2 * NX0_3D - 1, "ny": 9, "nz": 9},
    "fine":     {"nx": 4 * NX0_3D - 3, "ny": 9, "nz": 9},
}
cases_3d = {}
for lvl, dic in meshes_3d.items():
    dd = {
        "dim": 3,
        "origine": "0.  0.  0.",
        "nodes": f"{dic['nx']} {dic['ny']} {dic['nz']}",
        "lengths": "1.  1.  1.",
        "borders": (
            "Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.\n"
            "        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.\n"
            "        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.\n"
            "        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.\n"
            "        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.\n"
            "        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1."
        ),
        "polyedriser": "Polyedriser dom",
        "nvel": 3,
        "vel": "0.2 0. 0.",
        "bc": (
            "gauche   Neumann_Paroi_adiabatique\n"
            "            droite   Neumann_Paroi_adiabatique\n"
            "            bas      Neumann_Paroi_adiabatique\n"
            "            haut     Neumann_Paroi_adiabatique\n"
            "            devant   Neumann_Paroi_adiabatique\n"
            "            derriere Neumann_Paroi_adiabatique"
        ),
        "probes": (
            "sonde_x015 grav magnetic_field Periode 1e-6 Points 1 0.15 0.5 0.5\n"
            "      sonde_x025 grav magnetic_field Periode 1e-6 Points 1 0.25 0.5 0.5\n"
            "      sonde_x045 grav magnetic_field Periode 1e-6 Points 1 0.45 0.5 0.5"
        ),
    }
    cases_3d[lvl] = run.addCaseFromTemplate("advection.data", f"advection_3d/{lvl}",
                                            dic=dd, nbProcs=NB_PROCS, excluNR=True)
    cases_3d[lvl].partition()

run.printCases()
run.runCases()

case_2d = cases_2d["coarse"]
case_3d = cases_3d["coarse"]


## Checks


In [ ]:
for label, case in (("2D", case_2d), ("3D", case_3d)):
    case_dir = f"advection_{label.lower()}/coarse"
    t, b_peak0 = read_son_dir(case_dir, case.dataFileName_, "SONDE_X025")
    t2, b_trav = read_son_dir(case_dir, case.dataFileName_, "SONDE_X045")
    peak0 = b_peak0[0]
    print(f"[{label}] initial peak b(x0,0) : {peak0:.4f}")
    print(f"[{label}] b at x0 final time : {b_peak0[-1]:.4f} (bump has left)")
    print(f"[{label}] b at x0+Ut : {b_trav[0]:.4f} -> {b_trav[-1]:.4f}")
    assert b_peak0[-1] < 0.3, f"[{label}] bump did not leave x0: {b_peak0[-1]:.3f}"
    assert b_trav[0] < 0.3, f"[{label}] bump present x0+Ut at t=0: {b_trav[0]:.3f}"
    ratio = b_trav[-1] / peak0
    print(f"[{label}] arrival peak / initial peak : {ratio:.3f}")
    assert ratio > 0.5, f"[{label}] peak not transported: arrival {b_trav[-1]:.3f} < 0.5*{peak0:.3f}"
    print(f"OK [{label}]: bump advected right; peak preserved within amont diffusion.")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(t, b_peak0, label=f"[{label}] b(x0,t)")
    plt.plot(t2, b_trav, label=f"[{label}] b(x0+Ut,t)")
    plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()


## Mesh convergence

The shared refinement ladder is adapted to the 1D transport physics: only the
active `x` direction is refined. In 2D the levels are `65×17`, `129×17`, and
`257×17`; in 3D they are `65×9×9`, `129×9×9`, and `257×9×9`. The transverse
directions remain coarse and uniform.

For both dimensions, the metric is the missing peak at the arrival probe
$x_0 + Ut = 0.45$ at $t_{\max}$, $e_h = 1 - b_h(0.45, t_{\max})$ (the
analytic peak is 1 — pure advection does not damp). Both the implicit-Euler
temporal damping (fixed `dt` at every level) and the `amont` upwind numerical
diffusion reduce the peak monotonically, so a strict decrease of $e_h$ across
levels is structural; the level-to-level gain is dominated by the first-order
upwind spatial error, and the observed order is $\log_2(e_k/e_{k+1})$.


In [ ]:
SONDE_ARRIVAL = "SONDE_X045"

def arrival_deficit(subdir, case_name):
    _, b = read_son_dir(subdir, case_name, SONDE_ARRIVAL)
    return 1.0 - float(b[-1])

for dim_label, cases, scan_nodes in (
    ("2D", cases_2d, {"coarse": (65, 17), "standard": (129, 17), "fine": (257, 17)}),
    ("3D", cases_3d, {"coarse": (65, 9, 9), "standard": (129, 9, 9), "fine": (257, 9, 9)}),
):
    print(f"=== {dim_label} mesh convergence ===")
    scan_err = {
        lvl: arrival_deficit(f"advection_{dim_label.lower()}/{lvl}", cases[lvl].dataFileName_)
        for lvl in ("coarse", "standard", "fine")
    }
    prev = None
    for lvl in ("coarse", "standard", "fine"):
        nodes = scan_nodes[lvl]
        e = scan_err[lvl]
        order = "--"
        if prev is not None and prev > 0 and e > 0:
            order = f"{np.log2(prev / e):.2f}"
        print(f"{lvl:<9} {str(nodes):<14} {e:<26.4e} {order}")
        prev = e
    assert scan_err["standard"] < scan_err["coarse"], f"[{dim_label}] no convergence coarse -> standard"
    assert scan_err["fine"] < scan_err["standard"], f"[{dim_label}] no convergence standard -> fine"
    print(f"OK [{dim_label}]: convergence confirmed.")


## Computer Performance


In [ ]:
run.tablePerf()